# **🔬 Clean & Jerk Dataset — Individual Set Inspector**

### 🔧 Colab Setup & Library Init

In [1]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import DEFAULT_CLEAN_DATASET_DIR, load_summary_tables, load_set_h5

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
print(f"📂 Dataset folder: {dataset_dir}")

📂 Dataset folder: /content/drive/MyDrive/Academic/Research/Wearable Sensing/Wearable Biomechanics/Clean&JerkDataset


### 🎯 Select & Load a Set (`.h5`)

In [8]:
# @title 🎯 Load Set HDF5 { display-mode: "form" }
import pandas as pd

set_id = "set_0020" # @param {type:"string"}
set_data = load_set_h5(dataset_dir, set_id)

print(f"✅ Loaded {set_id}: {set_data['accel'].shape[0]} samples ({set_data['time_s'][-1]:.2f}s) | {len(set_data['sensors'])} IMUs")
df_set_meta = pd.DataFrame([set_data["metadata"]])
df_part_meta = pd.DataFrame([set_data["participant_metadata"]])
display_table(df_set_meta, clean_columns=False)
display_table(df_part_meta, clean_columns=False)

✅ Loaded set_0020: 6247 samples (62.46s) | 26 IMUs


   duration_s  has_faults  imu_fs_hz  imu_samples  lifts_total  lifts_usable  \
0       62.47       False      100.0         6247            5             5   

   load_kg participant_id  sensors_ok  sensors_total session_date  \
0     20.0          P0010          26             26   2026-06-10   

  session_time    set_id  
0  10:49+05:30  set_0020  

  achievement  age_years  best_weight_kg  bmi_kg_m2  body_mass_kg  \
0          NA         24            50.0      21.78          69.0   

   experience_years  height_m  \
0               5.0      1.78   

                                             notes participant_id sex  
0  Participation in Other sports: Rugby, Badminton          P0010   M  

### 🩺 26-IMU Sensor Status

In [9]:
# @title 🩺 26-IMU Sensor Status
from src.explorer import plot_set_sensor_status

plot_set_sensor_status(set_data).show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### ⚠️ Set Fault Log (`/fault_log`)

In [10]:
# @title ⚠️ Set Fault Log
df_set_faults = set_data["fault_log_df"]
display_table(df_set_faults, clean_columns=False)

No records found.


### ⏱️ Phase Markers — Consensus & Per-Annotator

In [11]:
# @title ⏱️ Phase Markers
df_consensus = set_data["consensus_df"]
df_annotators = set_data["annotators_df"]
print(f"Consensus markers: {len(df_consensus)} rows | Individual annotator markers: {len(df_annotators)} rows")
display_table(df_consensus, clean_columns=False)
display_table(df_annotators, clean_columns=False)

Consensus markers: 60 rows | Individual annotator markers: 180 rows


    frame  imu_idx  lift_id        marker_name  mean_frame  n_annotators  \
0       0        0        1              start      0.0000             3   
1     242      807        1              setup    241.6667             3   
2     285      950        1         first_pull    285.3333             3   
3     312     1040        1        second_pull    311.6667             3   
4     316     1053        1              clean    316.0000             3   
5     329     1097        1  intermediate_rest    328.6667             3   
6     332     1107        1         jerk_split    331.6667             3   
7     365     1217        1      jerk_recovery    364.6667             3   
8     395     1317        1      overhead_hold    395.3333             3   
9     433     1443        1            descend    432.6667             3   
10    518     1727        1          post_rest    518.3333             3   
11    582     1940        1                end    582.0000             3   
12    518   

    annotator_id  frame  imu_idx  lift_id    marker_name     time_s
0             a7      0        0        1          start   0.000000
1             a7    244      813        1          setup   8.133333
2             a7    285      950        1     first_pull   9.500000
3             a7    313     1043        1    second_pull  10.433333
4             a7    318     1060        1          clean  10.600000
..           ...    ...      ...      ...            ...        ...
175           a9   1647     5490        5  jerk_recovery  54.900000
176           a9   1684     5613        5  overhead_hold  56.133333
177           a9   1694     5647        5        descend  56.466667
178           a9   1786     5953        5      post_rest  59.533333
179           a9   1803     6010        5            end  60.100000

[180 rows x 6 columns]

### 📈 IMU Signal + Phase Markers Overlay

In [12]:
# @title 📈 IMU Signal + Phase Markers Overlay { display-mode: "form" }
from src.explorer import plot_set_imu_with_markers

sensor_code = "RKP" # @param ["RKP","RHD","REP","RSD","LKP","LHD","LEP","LSD","LHP","RHP","LS","RWD","RWP","RED","LWD","LWP","LED","LAD","LAP","LKD","LSP","RSP","US","RAD","RAP","RKD"]
lift_id = 0 # @param {type:"integer"}  # 0 = All Lifts

fig_imu = plot_set_imu_with_markers(
    set_data,
    sensor_code=sensor_code,
    lift_id=None if lift_id == 0 else lift_id,
)
fig_imu.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### 📋 Full IMU DataFrame (`/imu_data`)

In [ ]:
# @title 📋 Full IMU DataFrame
df_imu = set_data["imu_df"]
display_table(df_imu.head(200), clean_columns=False)

### 🎥 Set Videos (`/videos/set_XXXX_*.mp4`)

In [ ]:
# @title 🎥 Preview Set Video { display-mode: "form" }
from IPython.display import Video, display

camera_view = "front" # @param ["front", "side"]
vid_path = os.path.join(dataset_dir, "videos", f"{set_id}_{camera_view}.mp4")
if os.path.exists(vid_path):
    print(f"🎬 {vid_path} ({os.path.getsize(vid_path)/1e6:.1f} MB)")
    display(Video(vid_path, width=640, embed=False))
else:
    print(f"ℹ️ No {camera_view} video found at {vid_path}")